# Battery Cycle‑Life Analyzer — Demo

Fit degradation models, project remaining useful life, and visualise results.

```
pip install -e .
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bcla import core, viz, datasets

## 1. Load synthetic LFP cycling data

In [ ]:
cycles, capacity = datasets.synthetic_lfp(cycles=1500, seed=42)
print(f"Cycles: {len(cycles)}, capacity range: [{capacity.min():.4f}, {capacity.max():.4f}]")

## 2. Fit all degradation models

In [ ]:
results = core.fit_all_models(cycles, capacity)
for name, r in results.items():
    print(r.summary() + "\n")

## 3. Best model & EOL projection

In [ ]:
name, best = core.best_model(results, criterion="rmse")
print(f"Best model: {name} (R² = {best.r_squared:.4f})")
eol = best.eol_cycle(eol_fraction=0.8)
print(f"Projected EOL (80%): {eol:.0f} cycles" if eol else "EOL not reached within range")

## 4. Visualise fitted curves

In [ ]:
fig = viz.model_comparison(results)
fig.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Temperature effect on cycle life

In [ ]:
fig2, ax = plt.subplots(figsize=(8, 4.5))
viz.eol_vs_temperature(q0=1.0, k=0.00018, temperatures=[15, 25, 35, 45, 55], ax=ax)
fig2.savefig("temperature_effect.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Compare LFP vs NMC chemistry

In [ ]:
x_lfp, y_lfp = datasets.synthetic_lfp(cycles=1200)
x_nmc, y_nmc = datasets.synthetic_nmc(cycles=1200)

fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
r_lfp = core.fit_capacity_fade(x_lfp, y_lfp, model="power_law")
r_nmc = core.fit_capacity_fade(x_nmc, y_nmc, model="linear")
viz.capacity_fade(r_lfp, ax=ax1, title="LFP (power‑law fit)")
viz.capacity_fade(r_nmc, ax=ax2, title="NMC (linear fit)")
fig3.suptitle("Chemistry Comparison", fontsize=14, y=1.03)
fig3.tight_layout()
fig3.savefig("chemistry_comparison.png", dpi=150, bbox_inches="tight")
plt.show()